# Prerequisites: Set up MLflow and Pytorch

In [1]:
pip install mlflow torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.8/899.8 MB 52.6 MB/s  0:00:16m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 53.7 MB/s  0:00:10m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 48.1 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 55.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 46.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 53.3 MB/s  0:00:12m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 54.9 MB/s  0:00:03m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 49.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 54.1 MB/s  0:00:01 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 55.5 MB/s  0:00:04m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 55.3 MB/s  0:00:05m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

# Step 1: Create a new experiment

In [2]:
import mlflow

# The set_experiment API creates a new experiment if it doesn't exist.
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("Deep Learning Experiment")

# IMPORTANT: Enable system metrics monitoring
mlflow.config.enable_system_metrics_logging()
mlflow.config.set_system_metrics_sampling_interval(1)

2025/12/29 18:07:33 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 18:07:33 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 18:07:33 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 18:07:33 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 18:07:33 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 18:07:33 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 18:07:33 INFO mlflow.tracking.fluent: Experiment with name 'Deep Learning Experiment' does not exist. Creating a new experiment.


# Step 2: Prepare the dataset

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Define device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load and prepare data
transform = transforms.Compose(
    [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
)
train_dataset = datasets.FashionMNIST(
    "data", train=True, download=True, transform=transform
)
test_dataset = datasets.FashionMNIST("data", train=False, transform=transform)
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000)

100%|██████████| 26.4M/26.4M [00:08<00:00, 2.99MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 104kB/s]
100%|██████████| 4.42M/4.42M [00:02<00:00, 1.79MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 6.12MB/s]


# Step 3: Define the model and optimizer

In [4]:
import torch.nn as nn


class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


model = NeuralNetwork().to(device)

In [5]:
# Training parameters
params = {
    "epochs": 5,
    "learning_rate": 1e-3,
    "batch_size": 64,
    "optimizer": "SGD",
    "model_type": "MLP",
    "hidden_units": [512, 512],
}

# Define optimizer and loss function
loss_fn = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=params["learning_rate"])

# Step 4: Train the model

In [6]:
with mlflow.start_run() as run:
    # Log training parameters
    mlflow.log_params(params)

    for epoch in range(params["epochs"]):
        model.train()
        train_loss, correct, total = 0, 0, 0

        for batch_idx, (data, target) in enumerate(train_loader):
            data, target = data.to(device), target.to(device)

            # Forward pass
            optimizer.zero_grad()
            output = model(data)
            loss = loss_fn(output, target)

            # Backward pass
            loss.backward()
            optimizer.step()

            # Calculate metrics
            train_loss += loss.item()
            _, predicted = output.max(1)
            total += target.size(0)
            correct += predicted.eq(target).sum().item()

            # Log batch metrics (every 100 batches)
            if batch_idx % 100 == 0:
                batch_loss = train_loss / (batch_idx + 1)
                batch_acc = 100.0 * correct / total
                mlflow.log_metrics(
                    {"batch_loss": batch_loss, "batch_accuracy": batch_acc},
                    step=epoch * len(train_loader) + batch_idx,
                )

        # Calculate epoch metrics
        epoch_loss = train_loss / len(train_loader)
        epoch_acc = 100.0 * correct / total

        # Validation
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        with torch.no_grad():
            for data, target in test_loader:
                data, target = data.to(device), target.to(device)
                output = model(data)
                loss = loss_fn(output, target)

                val_loss += loss.item()
                _, predicted = output.max(1)
                val_total += target.size(0)
                val_correct += predicted.eq(target).sum().item()

        # Calculate and log epoch validation metrics
        val_loss = val_loss / len(test_loader)
        val_acc = 100.0 * val_correct / val_total

        # Log epoch metrics
        mlflow.log_metrics(
            {
                "train_loss": epoch_loss,
                "train_accuracy": epoch_acc,
                "val_loss": val_loss,
                "val_accuracy": val_acc,
            },
            step=epoch,
        )
        # Log checkpoint at the end of each epoch
        mlflow.pytorch.log_model(model, name=f"checkpoint_{epoch}")

        print(
            f"Epoch {epoch+1}/{params['epochs']}, "
            f"Train Loss: {epoch_loss:.4f}, Train Acc: {epoch_acc:.2f}%, "
            f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%"
        )

    # Log the final trained model
    model_info = mlflow.pytorch.log_model(model, name="final_model")

2025/12/29 18:09:40 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2025/12/29 18:09:40 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.


Epoch 1/5, Train Loss: 1.7638, Train Acc: 55.25%, Val Loss: 1.2593, Val Acc: 64.94%
Epoch 2/5, Train Loss: 1.0226, Train Acc: 67.93%, Val Loss: 0.8908, Val Acc: 69.70%
Epoch 3/5, Train Loss: 0.8110, Train Acc: 72.74%, Val Loss: 0.7685, Val Acc: 73.65%
Epoch 4/5, Train Loss: 0.7180, Train Acc: 75.93%, Val Loss: 0.6970, Val Acc: 76.23%
Epoch 5/5, Train Loss: 0.6563, Train Acc: 78.10%, Val Loss: 0.6461, Val Acc: 77.97%


2025/12/29 18:10:10 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/12/29 18:10:10 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


# Step 5: View the training results in the MLflow UI

In [9]:
!mlflow server --backend-store-uri sqlite:///mlflow.db --port 5000

Registry store URI not provided. Using backend store URI.
2025/12/29 18:14:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 18:14:34 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 18:14:34 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 18:14:34 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 18:14:34 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 18:14:34 INFO alembic.runtime.migration: Will assume non-transactional DDL.
2025/12/29 18:14:34 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2025/12/29 18:14:34 INFO mlflow.store.db.utils: Updating database tables
2025/12/29 18:14:34 INFO alembic.runtime.migration: Context impl SQLiteImpl.
2025/12/29 18:14:34 INFO alembic.runtime.migration: Will assume non-transactional DDL.
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --

# Step 6: Load back the model and run inference

In [8]:
# Load the final model
model = mlflow.pytorch.load_model("runs:/e179490878484d8c916386b1ca3502cb/final_model")
# or load a checkpoint
# model = mlflow.pytorch.load_model("runs:/<run_id>/checkpoint_<epoch>")
model.to(device)
model.eval()

# Resume the previous run to log test metrics
with mlflow.start_run(run_id=run.info.run_id) as run:
    # Evaluate the model on the test set
    test_loss, test_correct, test_total = 0, 0, 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
        output = model(data)
        loss = loss_fn(output, target)

        test_loss += loss.item()
        _, predicted = output.max(1)
        test_total += target.size(0)
        test_correct += predicted.eq(target).sum().item()

    # Calculate and log final test metrics
    test_loss = test_loss / len(test_loader)
    test_acc = 100.0 * test_correct / test_total

    mlflow.log_metrics({"test_loss": test_loss, "test_accuracy": test_acc})
    print(f"Final Test Accuracy: {test_acc:.2f}%")

/home/junspring/anaconda3/envs/mlflow/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025/12/29 18:14:27 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2025/12/29 18:14:27 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2025/12/29 18:14:27 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2025/12/29 18:14:27 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


Final Test Accuracy: 77.90%
